# Source-backed study-design audit

**Purpose:** reconstruct the independent biological units in the integrated four-dataset atlas before any statistical reanalysis.

This notebook is diagnostic and read-only with respect to the canonical analysis objects. It writes a corrected sample-design manifest and audit tables under:

`metadata/sensitivity/source_backed_study_design/`

The main issue tested here is whether short numeric DS3 patient identifiers were incorrectly merged across disease cohorts. The source publication describes GSE182109 as 44 tumor fragments from 18 distinct patients: 2 LGG, 11 newly diagnosed GBM, and 5 recurrent GBM. Ten patients underwent multiregional sampling. Therefore, `ndGBM-01` and `rGBM-01` are distinct patients even though both end in `01`.

This notebook:

1. reconstructs source-backed patient identifiers from complete DS3 sample labels;
2. distinguishes multiregional fragments from independent patients;
3. checks all source-publication numerical design checkpoints;
4. quantifies false collisions caused by numeric-only patient identifiers;
5. writes a model-planning table for the next analysis stage;
6. does not overwrite canonical metadata or analysis results.

Primary source evidence:

- Abdelfattah et al., *Nature Communications* 2022, DOI: `10.1038/s41467-022-28372-y`
- NCBI GEO accession: `GSE182109`


In [1]:
from pathlib import Path
import hashlib
import json
import os
import re
import sys
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)

NOTEBOOK_NAME = "02_source_backed_study_design_audit.ipynb"
OUT_REL = Path("metadata") / "sensitivity" / "source_backed_study_design"
INPUT_REL = Path("metadata") / "merged_metadata_cell_counts_by_sample_patient_condition.tsv"

SOURCE_DOI = "10.1038/s41467-022-28372-y"
SOURCE_PMC = "https://pmc.ncbi.nlm.nih.gov/articles/PMC8828877/"
SOURCE_GEO = "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE182109"

def resolve_project_root():
    candidates = []
    env_root = os.environ.get("GLIOMA_PROJECT_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())
    here = Path.cwd().resolve()
    candidates.extend([here, *here.parents])
    candidates.append(Path(r"G:\Repository\glioma-cnv-single-cell-atlas"))
    for candidate in candidates:
        if (candidate / INPUT_REL).exists() and (candidate / "code").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not resolve the repository root. Set GLIOMA_PROJECT_ROOT or run from the repository."
    )

ROOT = resolve_project_root()
INPUT = ROOT / INPUT_REL
OUT_DIR = ROOT / OUT_REL
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)
print("Input:", INPUT)
print("Output directory:", OUT_DIR)


Repository root: G:\Repository\glioma-cnv-single-cell-atlas
Input: G:\Repository\glioma-cnv-single-cell-atlas\metadata\merged_metadata_cell_counts_by_sample_patient_condition.tsv
Output directory: G:\Repository\glioma-cnv-single-cell-atlas\metadata\sensitivity\source_backed_study_design


## 1 Load and validate the canonical sample table

The input must contain exactly one row per canonical sample. This section checks that the expected 83 samples and four datasets are present before deriving any corrected identifiers.


In [2]:
sample = pd.read_csv(INPUT, sep="\t", dtype=str)
required = {
    "core_dataset", "geo_id", "sample_id", "patient_id", "condition",
    "diagnosis", "tumor_status", "n_cells"
}
missing = sorted(required - set(sample.columns))
if missing:
    raise AssertionError(f"Missing required columns: {missing}")

sample["n_cells"] = pd.to_numeric(sample["n_cells"], errors="raise").astype(np.int64)

if len(sample) != 83:
    raise AssertionError(f"Expected 83 sample rows, found {len(sample)}")
if sample["sample_id"].duplicated().any():
    dup = sample.loc[sample["sample_id"].duplicated(False), "sample_id"].tolist()
    raise AssertionError(f"Sample identifiers are not unique: {dup[:10]}")

expected_datasets = {
    "DS1_GSE103224": 8,
    "DS2_GSE141383": 9,
    "DS3_GSE182109": 44,
    "DS4_GSE278450": 22,
}
observed_datasets = sample.groupby("core_dataset").size().to_dict()
if observed_datasets != expected_datasets:
    raise AssertionError(f"Dataset sample counts differ: {observed_datasets}")

display(sample.head())
display(pd.Series(observed_datasets, name="n_samples").rename_axis("dataset").reset_index())


,core_dataset,geo_id,sample_id,patient_id,condition,diagnosis,tumor_status,n_cells
0,DS1_GSE103224,GSE103224,PJ016,PJ016,GBM,glioblastoma,gbm_unspecified,2838
1,DS1_GSE103224,GSE103224,PJ017,PJ017,GBM,glioblastoma,gbm_unspecified,1237
2,DS1_GSE103224,GSE103224,PJ018,PJ018,GBM,glioblastoma,gbm_unspecified,2163
3,DS1_GSE103224,GSE103224,PJ025,PJ025,GBM,glioblastoma,gbm_unspecified,5829
4,DS1_GSE103224,GSE103224,PJ030,PJ030,GBM,glioblastoma,gbm_unspecified,3048


,dataset,n_samples
0,DS1_GSE103224,8
1,DS2_GSE141383,9
2,DS3_GSE182109,44
3,DS4_GSE278450,22


## 2 Reconstruct source-backed patient and region identifiers

For DS3, the full biological patient label is the disease-cohort prefix plus its number, for example `ndGBM-01`, `rGBM-01`, or `LGG-03`. The trailing token identifies a sampled tumor region when present. For DS1, DS2, and DS4, the existing patient identifier is retained but scoped by dataset to avoid cross-study collisions.


In [3]:
DS3_PATTERN = re.compile(
    r"^(?:GSM\d+_)?(?P<cohort>ndGBM|rGBM|LGG)-(?P<patient_number>\d+)(?:-(?P<region>.+))?$"
)

def derive_design(row):
    dataset = str(row["core_dataset"])
    sample_id = str(row["sample_id"])
    old_patient = str(row["patient_id"])
    condition = str(row["condition"])

    if dataset == "DS3_GSE182109":
        match = DS3_PATTERN.match(sample_id)
        if match is None:
            raise AssertionError(f"Unrecognized DS3 sample label: {sample_id}")
        cohort = match.group("cohort")
        patient_number = match.group("patient_number")
        region = match.group("region")
        source_patient = f"{cohort}-{patient_number}"
        if cohort != condition:
            raise AssertionError(
                f"DS3 condition prefix mismatch for {sample_id}: prefix={cohort}, condition={condition}"
            )
        return pd.Series({
            "source_patient_label": source_patient,
            "source_region_label": region if region is not None else "single_region",
            "source_patient_number": patient_number,
            "patient_id_method": "DS3 full cohort-prefixed sample label",
        })

    return pd.Series({
        "source_patient_label": old_patient,
        "source_region_label": "single_region",
        "source_patient_number": old_patient,
        "patient_id_method": "canonical patient label scoped by dataset",
    })

design = pd.concat([sample.copy(), sample.apply(derive_design, axis=1)], axis=1)
design["source_backed_patient_id"] = (
    design["core_dataset"].astype(str) + "::" + design["source_patient_label"].astype(str)
)
design["patient_condition_id"] = (
    design["source_backed_patient_id"] + "::" + design["condition"].astype(str)
)
design["is_multiregional_patient"] = design.groupby("source_backed_patient_id")["sample_id"].transform("size") > 1
design["n_regions_for_patient"] = design.groupby("source_backed_patient_id")["sample_id"].transform("size").astype(int)
design["legacy_dataset_patient_id"] = (
    design["core_dataset"].astype(str) + "::" + design["patient_id"].astype(str)
)
design["legacy_patient_id_collision"] = (
    design.groupby("legacy_dataset_patient_id")["source_backed_patient_id"].transform("nunique") > 1
)

ordered = [
    "core_dataset", "geo_id", "sample_id", "condition", "diagnosis", "tumor_status",
    "n_cells", "patient_id", "legacy_dataset_patient_id", "source_patient_label",
    "source_backed_patient_id", "patient_condition_id", "source_region_label",
    "n_regions_for_patient", "is_multiregional_patient", "legacy_patient_id_collision",
    "patient_id_method"
]
design = design[ordered].sort_values(
    ["core_dataset", "condition", "source_patient_label", "sample_id"]
).reset_index(drop=True)

display(design[design["core_dataset"].eq("DS3_GSE182109")])


,core_dataset,geo_id,sample_id,condition,diagnosis,tumor_status,n_cells,patient_id,legacy_dataset_patient_id,source_patient_label,source_backed_patient_id,patient_condition_id,source_region_label,n_regions_for_patient,is_multiregional_patient,legacy_patient_id_collision,patient_id_method
17,DS3_GSE182109,GSE182109,GSM5518638_LGG-03,LGG,lower_grade_glioma,lower_grade_glioma,6833,03,DS3_GSE182109::03,LGG-03,DS3_GSE182109::LGG-03,DS3_GSE182109::LGG-03::LGG,single_region,1,False,True,DS3 full cohort-prefixed sample label
18,DS3_GSE182109,GSE182109,GSM5518630_LGG-04-1,LGG,lower_grade_glioma,lower_grade_glioma,2916,04,DS3_GSE182109::04,LGG-04,DS3_GSE182109::LGG-04,DS3_GSE182109::LGG-04::LGG,1,3,True,True,DS3 full cohort-prefixed sample label
19,DS3_GSE182109,GSE182109,GSM5518631_LGG-04-2,LGG,lower_grade_glioma,lower_grade_glioma,1492,04,DS3_GSE182109::04,LGG-04,DS3_GSE182109::LGG-04,DS3_GSE182109::LGG-04::LGG,2,3,True,True,DS3 full cohort-prefixed sample label
20,DS3_GSE182109,GSE182109,GSM5518632_LGG-04-3,LGG,lower_grade_glioma,lower_grade_glioma,2246,04,DS3_GSE182109::04,LGG-04,DS3_GSE182109::LGG-04,DS3_GSE182109::LGG-04::LGG,3,3,True,True,DS3 full cohort-prefixed sample label
21,DS3_GSE182109,GSE182109,GSM5518600_ndGBM-01-A,ndGBM,glioblastoma,newly_diagnosed,4631,01,DS3_GSE182109::01,ndGBM-01,DS3_GSE182109::ndGBM-01,DS3_GSE182109::ndGBM-01::ndGBM,A,4,True,True,DS3 full cohort-prefixed sample label
22,DS3_GSE182109,GSE182109,GSM5518601_ndGBM-01-C,ndGBM,glioblastoma,newly_diagnosed,9470,01,DS3_GSE182109::01,ndGBM-01,DS3_GSE182109::ndGBM-01,DS3_GSE182109::ndGBM-01::ndGBM,C,4,True,True,DS3 full cohort-prefixed sample label
23,DS3_GSE182109,GSE182109,GSM5518602_ndGBM-01-D,ndGBM,glioblastoma,newly_diagnosed,8770,01,DS3_GSE182109::01,ndGBM-01,DS3_GSE182109::ndGBM-01,DS3_GSE182109::ndGBM-01::ndGBM,D,4,True,True,DS3 full cohort-prefixed sample label
24,DS3_GSE182109,GSE182109,GSM5518603_ndGBM-01-F,ndGBM,glioblastoma,newly_diagnosed,4962,01,DS3_GSE182109::01,ndGBM-01,DS3_GSE182109::ndGBM-01,DS3_GSE182109::ndGBM-01::ndGBM,F,4,True,True,DS3 full cohort-prefixed sample label
25,DS3_GSE182109,GSE182109,GSM5518608_ndGBM-02-1,ndGBM,glioblastoma,newly_diagnosed,7436,02,DS3_GSE182109::02,ndGBM-02,DS3_GSE182109::ndGBM-02,DS3_GSE182109::ndGBM-02::ndGBM,1,4,True,True,DS3 full cohort-prefixed sample label
26,DS3_GSE182109,GSE182109,GSM5518609_ndGBM-02-2,ndGBM,glioblastoma,newly_diagnosed,5728,02,DS3_GSE182109::02,ndGBM-02,DS3_GSE182109::ndGBM-02,DS3_GSE182109::ndGBM-02::ndGBM,2,4,True,True,DS3 full cohort-prefixed sample label


## 3 Audit false patient collisions

Numeric-only DS3 identifiers such as `01`, `02`, `03`, `04`, and `05` occur in more than one disease cohort. This section records every collision and shows which distinct source patients would be merged by the legacy identifier.


In [4]:
collision_rows = []
for legacy_id, group in design.groupby("legacy_dataset_patient_id", sort=True):
    source_patients = sorted(group["source_backed_patient_id"].unique())
    conditions = sorted(group["condition"].unique())
    if len(source_patients) > 1:
        collision_rows.append({
            "legacy_dataset_patient_id": legacy_id,
            "n_distinct_source_patients": len(source_patients),
            "source_backed_patient_ids": ";".join(source_patients),
            "conditions": ";".join(conditions),
            "n_samples_affected": int(len(group)),
            "n_cells_affected": int(group["n_cells"].sum()),
            "audit_interpretation": "false cross-cohort patient merge",
        })

collision_audit = pd.DataFrame(collision_rows)
if len(collision_audit) != 5:
    raise AssertionError(f"Expected five DS3 numeric patient collisions, found {len(collision_audit)}")

display(collision_audit)


,legacy_dataset_patient_id,n_distinct_source_patients,source_backed_patient_ids,conditions,n_samples_affected,n_cells_affected,audit_interpretation
0,DS3_GSE182109::01,2,DS3_GSE182109::ndGBM-01;DS3_GSE182109::rGBM-01,ndGBM;rGBM,8,54762,false cross-cohort patient merge
1,DS3_GSE182109::02,2,DS3_GSE182109::ndGBM-02;DS3_GSE182109::rGBM-02,ndGBM;rGBM,8,46750,false cross-cohort patient merge
2,DS3_GSE182109::03,3,DS3_GSE182109::LGG-03;DS3_GSE182109::ndGBM-03;...,LGG;ndGBM;rGBM,7,24702,false cross-cohort patient merge
3,DS3_GSE182109::04,3,DS3_GSE182109::LGG-04;DS3_GSE182109::ndGBM-04;...,LGG;ndGBM;rGBM,8,40706,false cross-cohort patient merge
4,DS3_GSE182109::05,2,DS3_GSE182109::ndGBM-05;DS3_GSE182109::rGBM-05,ndGBM;rGBM,4,15455,false cross-cohort patient merge


## 4 Source-publication study-design checkpoints

The source publication states that DS3 contains 44 tumor fragments from 18 distinct patients: 2 LGG, 11 ndGBM, and 5 rGBM. Ten patients had multiregional sampling. These are hard design checkpoints.


In [5]:
patient_summary = (
    design.groupby([
        "core_dataset", "source_backed_patient_id", "source_patient_label", "condition"
    ], as_index=False)
    .agg(
        n_samples=("sample_id", "nunique"),
        n_cells=("n_cells", "sum"),
        sample_ids=("sample_id", lambda x: ";".join(sorted(map(str, x)))),
        region_labels=("source_region_label", lambda x: ";".join(sorted(map(str, x)))),
    )
)
patient_summary["is_multiregional"] = patient_summary["n_samples"] > 1

ds3_patients = patient_summary[patient_summary["core_dataset"].eq("DS3_GSE182109")].copy()
ds3_condition_patients = ds3_patients.groupby("condition").size().to_dict()

checks = [
    ("all_sample_rows", 83, len(design)),
    ("all_unique_source_backed_patients", 57, design["source_backed_patient_id"].nunique()),
    ("all_unique_patient_condition_units", 57, design["patient_condition_id"].nunique()),
    ("patients_spanning_conditions", 0, int((design.groupby("source_backed_patient_id")["condition"].nunique() > 1).sum())),
    ("DS3_sample_fragments", 44, int((design["core_dataset"] == "DS3_GSE182109").sum())),
    ("DS3_unique_patients", 18, ds3_patients["source_backed_patient_id"].nunique()),
    ("DS3_LGG_patients", 2, ds3_condition_patients.get("LGG", 0)),
    ("DS3_ndGBM_patients", 11, ds3_condition_patients.get("ndGBM", 0)),
    ("DS3_rGBM_patients", 5, ds3_condition_patients.get("rGBM", 0)),
    ("DS3_multiregional_patients", 10, int(ds3_patients["is_multiregional"].sum())),
    ("legacy_false_collision_groups", 5, len(collision_audit)),
]

source_checkpoint = pd.DataFrame(checks, columns=["check", "expected", "observed"])
source_checkpoint["pass"] = source_checkpoint["expected"] == source_checkpoint["observed"]
source_checkpoint["source"] = np.where(
    source_checkpoint["check"].str.startswith("DS3_"),
    "Abdelfattah et al. 2022 and GEO GSE182109",
    "Four-dataset canonical sample manifest plus source-backed DS3 reconstruction"
)

display(source_checkpoint)
if not source_checkpoint["pass"].all():
    raise AssertionError("One or more source-backed design checkpoints failed")


,check,expected,observed,pass,source
0,all_sample_rows,83,83,True,Four-dataset canonical sample manifest plus so...
1,all_unique_source_backed_patients,57,57,True,Four-dataset canonical sample manifest plus so...
2,all_unique_patient_condition_units,57,57,True,Four-dataset canonical sample manifest plus so...
3,patients_spanning_conditions,0,0,True,Four-dataset canonical sample manifest plus so...
4,DS3_sample_fragments,44,44,True,Abdelfattah et al. 2022 and GEO GSE182109
5,DS3_unique_patients,18,18,True,Abdelfattah et al. 2022 and GEO GSE182109
6,DS3_LGG_patients,2,2,True,Abdelfattah et al. 2022 and GEO GSE182109
7,DS3_ndGBM_patients,11,11,True,Abdelfattah et al. 2022 and GEO GSE182109
8,DS3_rGBM_patients,5,5,True,Abdelfattah et al. 2022 and GEO GSE182109
9,DS3_multiregional_patients,10,10,True,Abdelfattah et al. 2022 and GEO GSE182109


## 5 Final independent-unit counts

The corrected design contains 57 independent patients. DS3 regional fragments are repeated observations within a patient and condition, not separate patients. There are no source-supported cross-condition patient pairs in DS3.


In [6]:
condition_counts = (
    design.groupby("condition", as_index=False)
    .agg(
        n_samples=("sample_id", "nunique"),
        n_patients=("source_backed_patient_id", "nunique"),
        n_cells=("n_cells", "sum"),
        n_multiregional_patients=("source_backed_patient_id", lambda x: int(
            patient_summary.set_index("source_backed_patient_id").loc[pd.unique(x), "is_multiregional"].sum()
        )),
    )
    .sort_values("condition")
    .reset_index(drop=True)
)

expected_condition_patients = {"GBM": 39, "LGG": 2, "ndGBM": 11, "rGBM": 5}
observed_condition_patients = condition_counts.set_index("condition")["n_patients"].to_dict()
if observed_condition_patients != expected_condition_patients:
    raise AssertionError(
        f"Corrected condition patient counts differ from expected: {observed_condition_patients}"
    )

display(condition_counts)
display(ds3_patients.sort_values(["condition", "source_patient_label"]))


,condition,n_samples,n_patients,n_cells,n_multiregional_patients
0,GBM,39,39,132692,0
1,LGG,4,2,13487,1
2,ndGBM,22,11,112416,4
3,rGBM,18,5,92832,5


,core_dataset,source_backed_patient_id,source_patient_label,condition,n_samples,n_cells,sample_ids,region_labels,is_multiregional
17,DS3_GSE182109,DS3_GSE182109::LGG-03,LGG-03,LGG,1,6833,GSM5518638_LGG-03,single_region,False
18,DS3_GSE182109,DS3_GSE182109::LGG-04,LGG-04,LGG,3,6654,GSM5518630_LGG-04-1;GSM5518631_LGG-04-2;GSM551...,1;2;3,True
19,DS3_GSE182109,DS3_GSE182109::ndGBM-01,ndGBM-01,ndGBM,4,27833,GSM5518600_ndGBM-01-A;GSM5518601_ndGBM-01-C;GS...,A;C;D;F,True
20,DS3_GSE182109,DS3_GSE182109::ndGBM-02,ndGBM-02,ndGBM,4,25516,GSM5518608_ndGBM-02-1;GSM5518609_ndGBM-02-2;GS...,1;2;4;5,True
21,DS3_GSE182109,DS3_GSE182109::ndGBM-03,ndGBM-03,ndGBM,3,8995,GSM5518623_ndGBM-03-1;GSM5518624_ndGBM-03-2;GS...,1;2;3,True
22,DS3_GSE182109,DS3_GSE182109::ndGBM-04,ndGBM-04,ndGBM,1,7335,GSM5518633_ndGBM-04,single_region,False
23,DS3_GSE182109,DS3_GSE182109::ndGBM-05,ndGBM-05,ndGBM,1,6377,GSM5518634_ndGBM-05,single_region,False
24,DS3_GSE182109,DS3_GSE182109::ndGBM-06,ndGBM-06,ndGBM,1,3059,GSM5518635_ndGBM-06,single_region,False
25,DS3_GSE182109,DS3_GSE182109::ndGBM-07,ndGBM-07,ndGBM,1,8627,GSM5518636_ndGBM-07,single_region,False
26,DS3_GSE182109,DS3_GSE182109::ndGBM-08,ndGBM-08,ndGBM,1,5650,GSM5518637_ndGBM-08,single_region,False


## 6 Consequences for the previous sensitivity analysis

The earlier sensitivity notebook grouped by dataset, numeric patient identifier, **and condition**. Therefore, it did not combine `ndGBM-01` with `rGBM-01` inside a patient-condition profile. Its 57 patient-condition rows correspond to the 57 corrected independent patients.

However, the earlier statement that five patients span conditions was incorrect. They are five numeric-identifier collisions between distinct newly diagnosed and recurrent patients. The corrected interpretation is:

- 57 independent patients in the four-dataset atlas;
- 18 independent DS3 patients;
- 10 DS3 patients with multiregional fragments;
- no paired newly diagnosed/recurrent patients supported by the source study;
- patient-level aggregation across regional fragments remains the appropriate sensitivity unit.


In [7]:
previous_sensitivity_dir = ROOT / "metadata" / "sensitivity" / "patient_condition_sensitivity"
previous_units_file = previous_sensitivity_dir / "condition_independent_unit_counts.tsv"

previous_audit = []
if previous_units_file.exists():
    prior = pd.read_csv(previous_units_file, sep="\t")
    previous_audit.append({
        "item": "previous_sensitivity_output_found",
        "observed": True,
        "interpretation": str(previous_units_file.relative_to(ROOT)),
    })
else:
    prior = None
    previous_audit.append({
        "item": "previous_sensitivity_output_found",
        "observed": False,
        "interpretation": "Previous output not required for this notebook",
    })

previous_audit.extend([
    {
        "item": "previous_patient_condition_unit_count",
        "observed": int(design["patient_condition_id"].nunique()),
        "interpretation": "57 rows remain the correct independent-patient count because no source patient spans conditions",
    },
    {
        "item": "previous_claim_of_cross_condition_patients",
        "observed": 5,
        "interpretation": "Incorrect; these are numeric identifier collisions between distinct cohort-prefixed patients",
    },
    {
        "item": "correct_source_patients_spanning_conditions",
        "observed": 0,
        "interpretation": "No paired ndGBM-rGBM design is supported by the source study",
    },
])
previous_sensitivity_interpretation = pd.DataFrame(previous_audit)
display(previous_sensitivity_interpretation)


,item,observed,interpretation
0,previous_sensitivity_output_found,True,metadata\sensitivity\patient_condition_sensiti...
1,previous_patient_condition_unit_count,57,57 rows remain the correct independent-patient...
2,previous_claim_of_cross_condition_patients,5,Incorrect; these are numeric identifier collis...
3,correct_source_patients_spanning_conditions,0,No paired ndGBM-rGBM design is supported by th...


## 7 Model-planning recommendations

This table defines the next analysis stage. It deliberately separates technically possible models from claims that are well supported by the available independent-patient counts.


In [8]:
model_plan = pd.DataFrame([
    {
        "analysis_family": "Compartment and malignant-state composition",
        "independent_unit": "source-backed patient within condition",
        "aggregation": "sum feature and denominator counts across all regional fragments from the same patient",
        "candidate_model": "beta-binomial or binomial mixed/count model; patient-level nonparametric sensitivity",
        "dataset_term": "required for GBM-versus-DS3 contrasts when estimable",
        "pairing": "none across conditions",
        "main_limitation": "LGG has 2 patients; rGBM has 5 patients; condition is partly confounded with dataset",
        "recommended_claim_strength": "exploratory for four-condition inference; emphasize effect sizes and exact patient counts",
    },
    {
        "analysis_family": "Program scores",
        "independent_unit": "source-backed patient within condition",
        "aggregation": "cell-count-weighted patient mean across regional fragments",
        "candidate_model": "patient-level robust comparison or linear model with dataset term when estimable",
        "dataset_term": "required for cross-dataset GBM contrasts when estimable",
        "pairing": "none across conditions",
        "main_limitation": "small LGG and rGBM patient counts",
        "recommended_claim_strength": "descriptive/exploratory unless robust across sensitivity analyses",
    },
    {
        "analysis_family": "Overall malignant pseudobulk DE",
        "independent_unit": "source-backed patient within condition",
        "aggregation": "sum raw counts across regional fragments before normalization",
        "candidate_model": "edgeR quasi-likelihood, DESeq2, or limma-voom with patient-level libraries",
        "dataset_term": "include dataset only where design matrix is identifiable",
        "pairing": "none across conditions",
        "main_limitation": "LGG n=2 is not suitable for strong genome-wide inference; condition-dataset confounding",
        "recommended_claim_strength": "primary contrasts only when independently replicated; otherwise exploratory",
    },
    {
        "analysis_family": "Within-state pseudobulk DE",
        "independent_unit": "source-backed patient-state within condition",
        "aggregation": "sum raw counts across all qualifying regional fragments for patient and state",
        "candidate_model": "count-based model with minimum patient and count filters",
        "dataset_term": "include where identifiable",
        "pairing": "none across conditions",
        "main_limitation": "state-specific patient counts may be smaller than total condition counts",
        "recommended_claim_strength": "only report contrasts meeting predeclared patient-count thresholds",
    },
    {
        "analysis_family": "Condition-specific LIANA",
        "independent_unit": "not inherently patient-replicate inference in the current pooled run",
        "aggregation": "current runs pool cells within each condition",
        "candidate_model": "retain as candidate discovery; add patient prevalence or leave-one-patient-out robustness",
        "dataset_term": "interpret condition and dataset representation explicitly",
        "pairing": "none across conditions",
        "main_limitation": "ranked interaction counts depend on cell abundance and group representation",
        "recommended_claim_strength": "candidate axes, not differential signaling",
    },
])

display(model_plan)


,analysis_family,independent_unit,aggregation,candidate_model,dataset_term,pairing,main_limitation,recommended_claim_strength
0,Compartment and malignant-state composition,source-backed patient within condition,sum feature and denominator counts across all ...,beta-binomial or binomial mixed/count model; p...,required for GBM-versus-DS3 contrasts when est...,none across conditions,LGG has 2 patients; rGBM has 5 patients; condi...,exploratory for four-condition inference; emph...
1,Program scores,source-backed patient within condition,cell-count-weighted patient mean across region...,patient-level robust comparison or linear mode...,required for cross-dataset GBM contrasts when ...,none across conditions,small LGG and rGBM patient counts,descriptive/exploratory unless robust across s...
2,Overall malignant pseudobulk DE,source-backed patient within condition,sum raw counts across regional fragments befor...,"edgeR quasi-likelihood, DESeq2, or limma-voom ...",include dataset only where design matrix is id...,none across conditions,LGG n=2 is not suitable for strong genome-wide...,primary contrasts only when independently repl...
3,Within-state pseudobulk DE,source-backed patient-state within condition,sum raw counts across all qualifying regional ...,count-based model with minimum patient and cou...,include where identifiable,none across conditions,state-specific patient counts may be smaller t...,only report contrasts meeting predeclared pati...
4,Condition-specific LIANA,not inherently patient-replicate inference in ...,current runs pool cells within each condition,retain as candidate discovery; add patient pre...,interpret condition and dataset representation...,none across conditions,ranked interaction counts depend on cell abund...,"candidate axes, not differential signaling"


## 8 Write outputs and final readiness checks


In [9]:
outputs = {}

def write_tsv(df, name):
    path = OUT_DIR / name
    df.to_csv(path, sep="\t", index=False)
    outputs[name] = path
    return path

write_tsv(design, "source_backed_sample_design.tsv")
write_tsv(patient_summary.sort_values(["core_dataset", "condition", "source_patient_label"]), "source_backed_patient_summary.tsv")
write_tsv(ds3_patients.sort_values(["condition", "source_patient_label"]), "ds3_patient_region_summary.tsv")
write_tsv(collision_audit, "legacy_patient_id_collision_audit.tsv")
write_tsv(source_checkpoint, "source_design_checkpoint.tsv")
write_tsv(condition_counts, "condition_sample_patient_counts.tsv")
write_tsv(previous_sensitivity_interpretation, "previous_sensitivity_interpretation.tsv")
write_tsv(model_plan, "patient_level_model_plan.tsv")

source_evidence = pd.DataFrame([
    {
        "source": "Abdelfattah et al. 2022",
        "identifier": SOURCE_DOI,
        "url": SOURCE_PMC,
        "evidence": "44 tumor fragments from 18 glioma patients: 2 LGG, 11 ndGBM, and 5 rGBM; 10 multiregional patients",
    },
    {
        "source": "NCBI GEO",
        "identifier": "GSE182109",
        "url": SOURCE_GEO,
        "evidence": "44 sample labels retain cohort-prefixed patient and region identifiers",
    },
])
write_tsv(source_evidence, "source_evidence.tsv")

readiness = pd.DataFrame([
    ("canonical_sample_rows", 83, len(design)),
    ("corrected_unique_patients", 57, design["source_backed_patient_id"].nunique()),
    ("DS3_unique_patients", 18, ds3_patients["source_backed_patient_id"].nunique()),
    ("DS3_multiregional_patients", 10, int(ds3_patients["is_multiregional"].sum())),
    ("patients_spanning_conditions", 0, int((design.groupby("source_backed_patient_id")["condition"].nunique() > 1).sum())),
    ("source_checkpoint_all_pass", True, bool(source_checkpoint["pass"].all())),
    ("canonical_files_modified", False, False),
], columns=["check", "expected", "observed"])
readiness["pass"] = readiness["expected"] == readiness["observed"]
write_tsv(readiness, "readiness_checks.tsv")

run_info = {
    "notebook": NOTEBOOK_NAME,
    "timestamp": datetime.now().astimezone().isoformat(),
    "python": sys.version,
    "repository_root": str(ROOT),
    "input": str(INPUT.relative_to(ROOT)),
    "output_directory": str(OUT_DIR.relative_to(ROOT)),
    "source_doi": SOURCE_DOI,
    "source_geo": "GSE182109",
    "canonical_files_modified": False,
}
(OUT_DIR / "run_info.json").write_text(json.dumps(run_info, indent=2), encoding="utf-8")
outputs["run_info.json"] = OUT_DIR / "run_info.json"

manifest_rows = []
for name, path in sorted(outputs.items()):
    data = path.read_bytes()
    if path.suffix == ".tsv":
        frame = pd.read_csv(path, sep="\t")
        rows, columns = frame.shape
    else:
        rows, columns = np.nan, np.nan
    manifest_rows.append({
        "file": name,
        "relative_path": str(path.relative_to(ROOT)),
        "rows": rows,
        "columns": columns,
        "bytes": len(data),
        "sha256": hashlib.sha256(data).hexdigest(),
    })

manifest = pd.DataFrame(manifest_rows)
manifest_path = OUT_DIR / "output_manifest.tsv"
manifest.to_csv(manifest_path, sep="\t", index=False)

display(readiness)
display(manifest)

if not readiness["pass"].all():
    raise AssertionError("One or more final readiness checks failed")

print("SOURCE-BACKED STUDY-DESIGN AUDIT PASSED")
print("Correct total independent patients: 57")
print("Correct DS3 independent patients: 18")
print("DS3 multiregional patients: 10")
print("Source-supported patients spanning conditions: 0")
print("Outputs written to:", OUT_DIR)


,check,expected,observed,pass
0,canonical_sample_rows,83,83,True
1,corrected_unique_patients,57,57,True
2,DS3_unique_patients,18,18,True
3,DS3_multiregional_patients,10,10,True
4,patients_spanning_conditions,0,0,True
5,source_checkpoint_all_pass,True,True,True
6,canonical_files_modified,False,False,True


,file,relative_path,rows,columns,bytes,sha256
0,condition_sample_patient_counts.tsv,metadata\sensitivity\source_backed_study_desig...,4.0,5.0,143,494f6374ba0f0baaf49ad4e2c7b8799f9f61f9fa93ec04...
1,ds3_patient_region_summary.tsv,metadata\sensitivity\source_backed_study_desig...,18.0,9.0,2414,597c192f25b803ed551292b1e752154d5e8efb244118c4...
2,legacy_patient_id_collision_audit.tsv,metadata\sensitivity\source_backed_study_desig...,5.0,7.0,800,09bede8cd8203bc001ef0ba9d4b04ad704bb3444d961c0...
3,patient_level_model_plan.tsv,metadata\sensitivity\source_backed_study_desig...,5.0,8.0,2289,40fa5ee92c4ace9611f3b6cb3b4ce44af88d6e38c0952e...
4,previous_sensitivity_interpretation.tsv,metadata\sensitivity\source_backed_study_desig...,4.0,3.0,544,3c3dcc6f0c6f46345985b634b96567b11972716ec4121e...
5,readiness_checks.tsv,metadata\sensitivity\source_backed_study_desig...,7.0,4.0,298,df0d16c411c5966dca390b9260170f89ee3a8cbeabe5e4...
6,run_info.json,metadata\sensitivity\source_backed_study_desig...,NaN,NaN,570,2345b492abd64002b5dbcdd3bee7aa5575586f75153c18...
7,source_backed_patient_summary.tsv,metadata\sensitivity\source_backed_study_desig...,57.0,9.0,6296,94db622351b6618a5482e4c510e7e982f0b97bf6c60bc4...
8,source_backed_sample_design.tsv,metadata\sensitivity\source_backed_study_desig...,83.0,17.0,20158,3f92f230598315685a9bcf81c3bce1191b973570a1b116...
9,source_design_checkpoint.tsv,metadata\sensitivity\source_backed_study_desig...,11.0,5.0,1064,b61101e69f96c1b4af95abd2f052da4a5d78592b2b5c00...


SOURCE-BACKED STUDY-DESIGN AUDIT PASSED
Correct total independent patients: 57
Correct DS3 independent patients: 18
DS3 multiregional patients: 10
Source-supported patients spanning conditions: 0
Outputs written to: G:\Repository\glioma-cnv-single-cell-atlas\metadata\sensitivity\source_backed_study_design
